In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL21hdHRoZXdmaXNoZXIvZGV2L3Byb2plY3RzL2dhdXNzZWQvR2F1c3NFRC9kb2NzL2V4YW1wbGVz'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/_bootstrap.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/_bootstrap_external.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/zipimport.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/codecs.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/aliases.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/__init__.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/encodings/utf_8.py": 1738680669.0, "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.

In [2]:
#| echo: False
import jax
from jax import config
import jax.numpy as jnp
from jax import Array
import numpy as np
config.update("jax_enable_x64", True)  # do this at the very top

import matplotlib.pyplot as plt

from tueplots import bundles, figsizes

plt.rcParams.update(bundles.probnum2025())
plt.rcParams.update(figsizes.probnum2025_half())

# plt.rcParams["figure.figsize"] = (5, 3)
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]  # matches LaTeX default


from gaussed.domains import Euclidean
from gaussed.domains.index import IndexDomain
from gaussed.codomains import Codomain

from gaussed.gp.gp_ops.base import KernelSpec, FunSpec
from gaussed.gp.gp_ops.point_eval import Eval
from gaussed.gp.gp_ops.partial import Partial
from gaussed.gp.gp_ops.grad import Grad
from gaussed.gp.gp_ops.affine import Affine
from gaussed.gp.gp_ops.weighted_eval import WeightedEval
from gaussed.gp.gp_ops.probe import Probe, ProbeStack, as_stack
from gaussed.gp.gp_ops.linearise import Linearise

from gaussed.gp.kernels.rbf import RBFKernel, RBFParams
from gaussed.gp.kernels.matern import MaternKernel, MaternParams
from gaussed.gp.kernels.imq import IMQKernel, IMQParams
from gaussed.gp.kernels.delta import DeltaKernel, DeltaParams
from gaussed.gp.kernels.stacked import StackedKernel
from gaussed.gp.kernels.coregional import CoregionParams, CoregionalKernel
from gaussed.gp.means import ZeroMeanFun
from gaussed.gp.base import GP, PosteriorGP

from gaussed.model import GPModel
from gaussed.utils.shape_helpers import pack_ev2mat

from gaussed.backends.base import ExactBackend
from gaussed.backends.solvers.linear_solver import SolverFns
from gaussed.backends.solvers.cholesky import (
    chol_solve_hook,
    chol_sqrt_hook,
    chol_logdet_hook,
)
from gaussed.backends.solvers.svd import svd_solve_hook, svd_sqrt_hook, svd_logdet_hook

from gaussed.likelihoods.gaussian import GaussianLikelihood
from gaussed.likelihoods.gaussian_noise import DiagonalNoise

solver_fns = SolverFns(svd_solve_hook(), svd_sqrt_hook(), svd_logdet_hook())

In [3]:
#| fig-align: center
#| out-width: 80%

# --- GP prior ---
domain   = Euclidean((1,))
codomain = Codomain((1,))
mean     = ZeroMeanFun(domain, codomain)
imq_params = IMQParams(lengthscale=jnp.array(0.5), amplitude=jnp.array(1.24751754), beta=jnp.array(1.0))
imq_kernel   = IMQKernel(imq_params)
backend = ExactBackend(solverfns=solver_fns)

gp = GP(domain, codomain, mean, imq_kernel, backend)

# --- Data ---
key = jax.random.PRNGKey(0)
Xtr = jnp.linspace(-3.0, 3.0, 5).reshape(-1, 1)
y_clean = jnp.sin(Xtr).squeeze()
noise = 0.04
y = y_clean + noise * jax.random.normal(key, shape=y_clean.shape)

# --- Probes ---
Ftr = Probe(ops=(Partial(0),), fnl=Eval(Xtr))
Xt = jnp.linspace(-5.0, 5.0, 300).reshape(-1, 1)
Fte = Probe(ops=(), fnl=Eval(Xt))

# --- Model / condition ---
lik = GaussianLikelihood(noise=DiagonalNoise(jnp.array([0.01])))
model = GPModel(gp=gp, likelihood=lik)
post = model.condition(Ftr, y)

# --- Prior mean/var (for reference) ---
prior_mean = gp.mean_spec().eval(Xt)[:, 0]
prior_var  = jnp.diag(pack_ev2mat(gp.kernel_spec().k0(Xt, Xt), (1,), (1,)))

# --- Posterior mean/var ---
m_post = post.mean(Fte)
v_post = post.variance(Fte)

# --- Plot ---
xplot = Xt.squeeze()

plt.figure(figsize=(10,5))

# Prior
plt.subplot(1,2,1)
plt.title("Prior GP (IMQ)")
plt.fill_between(xplot,
                 prior_mean - 2*jnp.sqrt(prior_var),
                 prior_mean + 2*jnp.sqrt(prior_var),
                 alpha=0.3, label="$\\pm2\\sigma$")
plt.plot(xplot, prior_mean, "k--", label="mean")
plt.scatter(Xtr.squeeze(), y, s=25, c="r", label="Data")
plt.legend()

# Posterior
plt.subplot(1,2,2)
plt.title("Posterior GP")
plt.fill_between(xplot,
                 m_post.reshape(-1,) - 2*jnp.sqrt(v_post),
                 m_post.reshape(-1,) + 2*jnp.sqrt(v_post),
                 alpha=0.3, label="$\\pm2\\sigma$")
plt.plot(xplot, m_post, label="mean")
plt.scatter(Xtr.squeeze(), y, s=25, c="r", label="Data")
plt.legend()

plt.tight_layout()
# mplcyberpunk.add_glow_effects()

plt.show()

/var/folders/y9/f2jq2rkn6h16kl4g9j02c2800000gn/T/ipykernel_3002/2933727991.py:63: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


<Figure size 3000x1500 with 2 Axes>

In [4]:
# ----------------------------
# GP prior (2D, with ARD)
# ----------------------------
domain   = Euclidean((2,))
codomain = Codomain((1,))
mean     = ZeroMeanFun(domain, codomain)

solver_fns = SolverFns(chol_solve_hook(), chol_sqrt_hook(), chol_logdet_hook())

backend = ExactBackend(solverfns=solver_fns)

# ARD lengthscales (ℓ_x, ℓ_y); amplitude = σ²
imq_params = IMQParams(
    lengthscale=jnp.array([1.0, 0.7]),  # ARD
    amplitude=jnp.array(1.0),
    beta=jnp.array(1.0),
)
kernel  = IMQKernel(imq_params)

gp      = GP(domain, codomain, mean, kernel, backend)

# ----------------------------
# Synthetic 2D data
# ----------------------------
key = jax.random.PRNGKey(0)
n_tr = 60
Xtr = jax.random.uniform(key, (n_tr, 2), minval=-3.0, maxval=3.0)

def f_true(xy: jnp.ndarray) -> jnp.ndarray:
    # smooth, non-separable target
    x, y = xy[..., 0], xy[..., 1]
    return jnp.sin(x) * jnp.cos(0.8 * y) + 0.2 * x

y_clean = jax.vmap(f_true)(Xtr)
noise   = 0.1
y       = y_clean + noise * jax.random.normal(key, shape=y_clean.shape)

# ----------------------------
# Probes
# ----------------------------
Ftr   = Probe(ops=(), fnl=Eval(Xtr))                 # function values at Xtr
lik   = GaussianLikelihood(noise=DiagonalNoise(jnp.array([0.01])))
model = GPModel(gp=gp, likelihood=lik)
post  = model.condition(Ftr, y)

# Test grid for visualisation
nx = ny = 60
x1 = jnp.linspace(-3.5, 3.5, nx)
x2 = jnp.linspace(-3.5, 3.5, ny)
Xg1, Xg2 = jnp.meshgrid(x1, x2, indexing="xy")
Xgrid = jnp.stack([Xg1.ravel(), Xg2.ravel()], axis=1)     # (nx*ny, 2)

F_eval  = Probe(ops=(), fnl=Eval(Xgrid))                  # function probe
F_grad  = Probe(ops=(Grad(), ), fnl=Eval(Xgrid))  # gradient probe (∂/∂x, ∂/∂y)

# Posterior mean and gradient mean
m_post_grid = post.mean(F_eval).reshape(nx*ny)            # (N,)
g_post_grid = post.mean(F_grad).reshape(nx*ny, 2)         # (N, 2)

In [5]:
#| fig-align: center
#| out-width: 75%
#| echo: false

# ----------------------------
# Plotting (two separate figures)
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

# 1) Posterior mean as filled contours
ax = axes[0]
M = m_post_grid.reshape(ny, nx)
cf = ax.contourf(Xg1, Xg2, M, levels=25)
ax.scatter(np.array(Xtr[:, 0]), np.array(Xtr[:, 1]), s=15, c="k", alpha=0.6)
ax.set_title("Posterior Mean")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")

# 2) Gradient quiver (posterior mean gradient)
ax = axes[1]
G = g_post_grid.reshape(ny, nx, 2)
step = max(1, nx // 20)  # thin arrows for clarity
ax.quiver(
    Xg1[::step, ::step], Xg2[::step, ::step],
    G[::step, ::step, 0], G[::step, ::step, 1],
    angles="xy", scale_units="xy", scale=1.0
)
ax.scatter(Xtr[:, 0], Xtr[:, 1], s=10)
ax.set_title("Posterior Mean Gradient")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
# ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
# mplcyberpunk.add_glow_effects()

plt.show()

/var/folders/y9/f2jq2rkn6h16kl4g9j02c2800000gn/T/ipykernel_3002/3575551168.py:31: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


<Figure size 2700x1350 with 2 Axes>

In [6]:
#| fig-align: center
#| out-width: 80%


# ----------------------------
# GP prior (2D, with ARD)
# ----------------------------
domain   = Euclidean((2,))
codomain = Codomain((1,))
mean     = ZeroMeanFun(domain, codomain)

solver_fns = SolverFns(svd_solve_hook(), svd_sqrt_hook(), svd_logdet_hook())
backend = ExactBackend(solverfns=solver_fns)

# ARD lengthscales (ℓ_x, ℓ_y); amplitude = σ²
imq_params = IMQParams(
    lengthscale=jnp.array([2., 1.9]),  # ARD
    amplitude=jnp.array(1.0),
    beta=jnp.array(0.01),
)
kernel  = IMQKernel(imq_params)
gp      = GP(domain, codomain, mean, kernel, backend)

# ----------------------------
# Synthetic 2D data
# ----------------------------
key = jax.random.PRNGKey(0)
n_tr = 50
Xtr = jax.random.uniform(key, (n_tr, 2), minval=-3.0, maxval=3.0)

def f_true(xy: jnp.ndarray) -> jnp.ndarray:
    x, y = xy[..., 0], xy[..., 1]
    return 0.2 * jnp.sin(x) * jnp.cos(0.8 * y) + 0.3 * x  # scalar

# --- gradient observations (condition on these!) ---
# g_true: (n_tr, 2) with columns [∂f/∂x, ∂f/∂y]
g_true = jax.vmap(jax.grad(f_true))(Xtr)

grad_noise = 0.1
key_g = jax.random.split(key, 2)[1]
g_obs = g_true + grad_noise * jax.random.normal(key_g, g_true.shape)  # (n_tr, 2)

# ----------------------------
# Probes (train on gradients)
# ----------------------------
# Gradient operator produces a 2-output (multioutput) GP at each location.
Ftr   = Probe(ops=(Grad(),), fnl=Eval(Xtr))        # gradient at Xtr, shape ~ (n_tr, 2)
# Per-output diagonal noise (broadcast across n_tr)
lik   = GaussianLikelihood(noise=DiagonalNoise(jnp.array([0.01, 0.01])))
model = GPModel(gp=gp, likelihood=lik)

# Condition on gradient observations
post  = model.condition(Ftr, g_obs)

# grid for visualisation
nx = ny = 60
x1 = jnp.linspace(-3.5, 3.5, nx)
x2 = jnp.linspace(-3.5, 3.5, ny)
Xg1, Xg2 = jnp.meshgrid(x1, x2, indexing="xy")
Xgrid = jnp.stack([Xg1.ravel(), Xg2.ravel()], axis=1)     # (nx*ny, 2)

F_eval = Probe(ops=(),        fnl=Eval(Xgrid))            # function values
F_grad = Probe(ops=(Grad(),), fnl=Eval(Xgrid))            # gradient values

# Posterior means
m_post_grid = post.mean(F_eval).reshape(nx*ny)            # (N,)
g_post_grid = post.mean(F_grad).reshape(nx*ny, 2)         # (N,2)

In [7]:
#| echo: False

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=False)

# 1) Posterior mean as filled contours
ax = axes[0]
M = np.array(m_post_grid).reshape(ny, nx)
cf = ax.contourf(np.array(Xg1), np.array(Xg2), M, levels=25)
ax.scatter(np.array(Xtr[:, 0]), np.array(Xtr[:, 1]), s=20, c="k", alpha=1.)

# overlay observed gradient arrows at training points
X = np.array(Xtr[:, 0])
Y = np.array(Xtr[:, 1])
U = np.array(g_obs[:, 0])   # ∂f/∂x
V = np.array(g_obs[:, 1])   # ∂f/∂y

ax.quiver(
    X, Y, U, V,
    angles="xy", scale_units="xy", scale=0.7,
    color="k",  zorder=3
)

ax.set_title("Posterior Mean (conditioned on $\\nabla f$)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal", adjustable="box")
fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)

# 2) Gradient quiver (posterior mean gradient)
ax = axes[1]
G = np.array(g_post_grid).reshape(ny, nx, 2)
step = max(1, nx // 20)  # thin arrows for clarity
ax.quiver(
    np.array(Xg1[::step, ::step]), np.array(Xg2[::step, ::step]),
    G[::step, ::step, 0], G[::step, ::step, 1],
    angles="xy", scale_units="xy", scale=1.1
)
ax.scatter(np.array(Xtr[:, 0]), np.array(Xtr[:, 1]), s=20, alpha=1.)
ax.set_title("Posterior Mean Gradient (conditioned on $\\nabla f$)")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_aspect("equal", adjustable="box")
# mplcyberpunk.add_glow_effects()

plt.show()

<Figure size 3000x1200 with 3 Axes>

In [8]:
#| echo: False
backend  = ExactBackend(solverfns=solver_fns)

In [9]:
#| fig-align: center
#| out-width: 80%

# --- GP prior ---
domain   = Euclidean((1,))
codomain = Codomain((1,))
mean     = ZeroMeanFun(domain, codomain)
rbf_params = RBFParams(lengthscale=jnp.array(0.1), amplitude=jnp.array(1.0))
kernel   = RBFKernel(rbf_params)

gp = GP(domain, codomain, mean, kernel, backend)

# --- Data ---
key = jax.random.PRNGKey(0)
Xtr = jnp.linspace(-3.0, 3.0, 10).reshape(-1, 1)
y_clean = jnp.sin(Xtr).squeeze()
noise = 0.1
y = y_clean + noise * jax.random.normal(key, shape=y_clean.shape)

# --- Probes ---
Ftr = Probe(ops=(), fnl=Eval(Xtr))
Xt  = jnp.linspace(-5.0, 5.0, 300).reshape(-1, 1)
Fte = Probe(ops=(), fnl=Eval(Xt))

# --- Model / condition ---
lik   = GaussianLikelihood()
model = GPModel(gp=gp, likelihood=lik)
post  = model.condition(Ftr, y)

# --- Prior mean/var (for reference) ---
prior_mean = gp.mean_spec().eval(Xt)[:, 0]
prior_var  = jnp.diag(pack_ev2mat(gp.kernel_spec().k0(Xt, Xt), (1,), (1,)))

# --- Posterior (unwarped) ---
m_post = post.mean(Fte)          # shape (n,1)
v_post = post.variance(Fte)      # shape (n,)

# f_0 function
def ref_fn(X: Array) -> Array:
    return post.mean(Probe(ops=(), fnl=Eval(X)))  

g = lambda u: 0.1 + u ** 2 / 2

# jacobian=None => AD fallback via jax.jacfwd(g)
op_pos = Linearise(forward=g, jacobian=None, ref_fn=ref_fn)

# Test probe that applies the linearised warp
Fte_pos = Probe(ops=(op_pos,), fnl=Eval(Xt))

# Posterior under linearised warp (still Gaussian thanks to affine map)
m_pos = post.mean(Fte_pos)       # (n,1)
v_pos = post.variance(Fte_pos)   # (n,)

In [10]:
#| echo: False

# --- Plot ---
xplot = Xt.squeeze()

plt.figure(figsize=(12,5))

# Prior
plt.subplot(1,3,1)
plt.title("Prior GP (RBF)")
plt.fill_between(xplot,
                 prior_mean - 2*jnp.sqrt(prior_var),
                 prior_mean + 2*jnp.sqrt(prior_var),
                 alpha=0.3, label="$\\pm 2\\sigma$")
plt.plot(xplot, prior_mean, "k--", label="Mean")
plt.scatter(Xtr.squeeze(), y, s=25, label="Data")
plt.legend()

# Posterior (unwarped)
plt.subplot(1,3,2)
plt.title("Posterior GP (latent f)")
plt.fill_between(xplot,
                 m_post.reshape(-1,) - 2*jnp.sqrt(v_post),
                 m_post.reshape(-1,) + 2*jnp.sqrt(v_post),
                 alpha=0.3, label="$\\pm 2\\sigma$")
plt.plot(xplot, m_post, label="Mean")
plt.scatter(Xtr.squeeze(), y, s=25, label="Data")
plt.legend()

# Posterior with linearised softplus warp
plt.subplot(1,3,3)
plt.title("Linearised $g$ Warp")
plt.fill_between(xplot,
                 m_pos.reshape(-1,) - 2*jnp.sqrt(v_pos),
                 m_pos.reshape(-1,) + 2*jnp.sqrt(v_pos),
                 alpha=0.3, label="$\\pm 2\\sigma$")
plt.plot(xplot, m_pos, label="Mean")
plt.scatter(Xtr.squeeze(), g(y), s=25, label="Data")
plt.legend()

plt.tight_layout()
plt.show()

/var/folders/y9/f2jq2rkn6h16kl4g9j02c2800000gn/T/ipykernel_3002/173714460.py:39: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


<Figure size 3600x1500 with 3 Axes>